In [ ]:
import numpy as np
import opendssdirect as dss
#-----
import yadi.dss.model as dss_model
import yadi.dss.sensitivity as dss_sens
import yadi.dss.qsts as dss_qsts
import imp
imp.reload(dss_qsts)
#-----
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
matplotlib.rc('text', usetex=True)
matplotlib.rc('text.latex', preamble=r'\usepackage{amsmath,amssymb}')
sns.set_theme(context='paper',style='ticks')

# Modeling Secondary Distribution Circuit Voltage Sensitivities with OpenDSS

## Step 0: Load the Three-Phase Secondary Test Circuit
### Secondary topology test case 

In [ ]:
cktfile = r"/home/sam/Research/mohca_cl/mohca_cl/ga_tech/secondary/SecondaryTestCircuit_modified/Master.DSS"

In [ ]:
time_step = '15m'
simulation_steps = 35040
qsts = dss_qsts.DSS_Timeseries(redirects=cktfile,time_step=time_step,simulation_steps=simulation_steps)
Ynet,vnet = Y = qsts.get_node_ybus(True)

In [ ]:
fig,axes = plt.subplots(ncols=2)
h1 = sns.heatmap(np.real(Ynet),ax=axes[0],center=0,cmap='mako',cbar=False)
h2 = sns.heatmap(np.imag(Ynet),ax=axes[1],center=0,cmap='mako')
axes[0].set_title(r"$\operatorname{Real}(\mathbf{Y})$")
axes[1].set_title(r"$\operatorname{Imag}(\mathbf{Y})$")

## Item 1: Construct the Voltage Sensitivity Matrices

In [ ]:
sens = dss_sens.DSS_Sensitivities(cktfile,verbose=False)

In [ ]:
svp = sens.get_svp()
svq = sens.get_svq()
svp_total,svq_total=svp,svq

In [ ]:
#Plot overall matrix
start_idx = 45 #Bus idx to start plot from
fig,axes = plt.subplots(ncols=2,constrained_layout=True,figsize=(7,7/1.62828))

#cbar_ax = fig.add_axes([.89, .3, .02, .4])
pal = sns.diverging_palette(30, 150, l=65, center="light", as_cmap=True)
#plt.suptitle("Secondary Topology Test Network (STN) Primary Sensitivity Matrices")
axes[0].set_title(r"$\frac{\partial \boldsymbol{v}}{\partial \boldsymbol{p}}$ Active Primary Sensitivity Matrix")
axes[1].set_title(r"$\frac{\partial \boldsymbol{v}}{\partial \boldsymbol{q}}$ Primary Sensitivity Matrix")

sns.heatmap(svp["matrix"][start_idx:,start_idx:],
    square=True,
    ax = axes[0],
    robust=True,
    center=0,
    linewidths=0.0005,
    linecolor='black',
    cmap=pal,
    #xticklabels=3,
    #yticklabels=3,
    cbar_kws= {
        'shrink':1,
        'fraction':0.05,
        'label':r"$\frac{\partial v_i}{\partial p_l}$ (V p.u./kW)",
        'drawedges':False
    }, 
    rasterized=True
)

sns.heatmap(svq["matrix"][start_idx:,start_idx:],
    square=True,
    ax = axes[1],
    robust=True,
    center=0,
    linewidths=0.0005,
    linecolor='black',
    cmap=pal,
    #xticklabels=3,
    #yticklabels=3,
    cbar_kws= {
        'shrink':1,
        'fraction':0.05,
        'label':r'$\frac{\partial v_i}{\partial q_l}$ (V p.u./kVAR)',
        'drawedges':False
    }, 
    rasterized=True
)
for ax in axes:
    ax.set_xlabel(r"Injection Node $l$")
    ax.set_ylabel(r"Measurement Node $i$")

#plt.savefig("/home/sam/research/mohca_cl/mohca_cl/ga_tech/secondary/Figures_Secondary/modified_primary_S.pdf",dpi=400)
#plt.savefig("/home/sam/research/mohca_cl/mohca_cl/ga_tech/secondary/Figures_Secondary/modified_primary_S.png",dpi=400)


In [ ]:
print(dss.Circuit.AllNodeNames())

In [ ]:
names = np.asarray(dss.Circuit.AllNodeNamesByPhase(Phase=3))
sec_names = []
for name in names:
    if "sec" in name:
        sec_names.append(name)
    if "trafo" in name:
        sec_names.append(name)
sec_names


In [ ]:
def make_secondary_slices(n_secondaries = 12):
    """
    Make slices of the node indexes of the currently loaded opendss feeder mapping to the secondary networks
    Params:
        n_secondaries: Number of secondary groups in the feeder
    """
    
    #Secondaries results dictionary
    secondaries = {}

    #Node names and node names by phase    
    node_names = dss.Circuit.AllNodeNames()
    a_node_names,b_node_names,c_node_names = dss.Circuit.AllNodeNamesByPhase(Phase=1),dss.Circuit.AllNodeNamesByPhase(Phase=2),dss.Circuit.AllNodeNamesByPhase(Phase=3)

    #Secondary indeces
    secondary_names = [str(i) for i in range(1,n_secondaries+1)] #Group index of the secondary names
    
    for sec_name in secondary_names:
        sec_bus_idx = 1 #Counter for the bus number within the secondary
        node_idxs_in_sec,node_names_in_sec = [],[] 
        
        for node_idx,node_name in enumerate(node_names):

            #Collect the phase info
            if(node_name in a_node_names):
                phase = 'a'
            elif(node_name in b_node_names):
                phase = 'b'
            elif(node_name in c_node_names):
                phase = 'c'
            else:
                print("no phase for: {node_name}".format(node_name=node_name))
                phase = ""

            if('sec' + sec_name + "_" in node_name): #Hacky way to separate by node name
                node_idxs_in_sec.append(node_idx)
                node_names_in_sec.append('s' + sec_name + "_" + str(sec_bus_idx) + "_"+phase)
                #Increment the secondary bus index counter
                sec_bus_idx += 1
            elif("trafo" + sec_name + "lv" in node_name): #Catch low voltage transformers
                node_idxs_in_sec.append(node_idx)
                node_names_in_sec.append('s' + sec_name + "_" + "xfmrlv"+ "_"+phase)
        secondaries['sec' + sec_name] = {
            "node_idxs":node_idxs_in_sec,
            "node_names":node_names_in_sec
        }
    return secondaries         


In [ ]:
names[-14:]

In [ ]:
secondaries = make_secondary_slices()
secondaries

## Item 2: Construct secondary matrices of varying topoologies FOR THE MODIFIED NETWORK

In [ ]:
#Make secondary submatrices
import pandas as pd
secondaries = make_secondary_slices()

#Note: hard code catch to catch the final subnetwork
names = dss.Circuit.AllNodeNames()
n_nodes = len(names)
sec_12_node_names = names[-14:]
n_nodes_sec_12 = len(sec_12_node_names)

sec_bus_idx = 1 #Counter for the bus number within the secondary
#Final sec12 network idxs
sec_12_node_idxs,formatted_sec_12_node_names = [],[]
for node_idx,node_name in enumerate(names):
    if(node_name in sec_12_node_names):
        sec_12_node_idxs.append(node_idx)
        if("xfmr" not in node_name):
            formatted_sec_12_node_names.append('s12_' + str(sec_bus_idx) + "_a")
            sec_bus_idx+=1
        else:
            formatted_sec_12_node_names.append("s12_xfmrlv_a")
secondaries["sec12"] = {
    'node_idxs':sec_12_node_idxs,
    'node_names':formatted_sec_12_node_names
}

In [ ]:
secondaries['sec12']

In [ ]:



svp0,svq0 = svp['matrix'],svq['matrix']

for S,suptitle,cbar_label in zip(
    [svp0,svq0],
    ["Active Power Secondary Sensitivity Submatrices","Reactive Power Secondary Sensitivity Submatrices"],
    [r'$\frac{\partial v_i}{\partial p_l}$ (V p.u./kW)',r'$\frac{\partial v_i}{\partial q_l}$ (V p.u./kVAR)']):
    
    
    #Make figure parameters
    #fig = plt.figure(figsize=(3*3.5,(2*3.5)/1.61828))
    fig,axes = plt.subplots(
        nrows=3,ncols=4,
        figsize=(2*3.5,(2*3.5)/1.61828),
        )
    #fig.subplots_adjust(hspace=0.55,wspace=0.45)


    #(nrows,ncols,plot_number)
    #subplot_params = [(3,4,1),(3,4,2),(3,4,3),(3,4,4),(3,4,5),(3,4,6),(3,4,7),(3,4,8),(3,5,12),(3,5,13),(3,5,14)]

    #Colorbar
    cbar_ax = fig.add_axes([.9, .3, .02, .4])
    #pal = sns.diverging_palette(30, 150, l=65, center="light", as_cmap=True)
    pal = sns.light_palette("seagreen", as_cmap=True)
    plt.suptitle(suptitle,fontsize=9)


    
    for i,(sec_name,d) in enumerate(secondaries.items()):
        #nrows,ncols,plot_number = subplot_params[i]

        #Setup axis of the subplot
        #ax = fig.add_subplot(nrows,ncols,plot_number)
        ax = axes.flatten()[i]
       
        #Make axis labels
        node_idxs = d["node_idxs"]
        node_names = d["node_names"]
        
        #Make dataframe
        S_sec = S[np.ix_(node_idxs,node_idxs)]
        S_sec = pd.DataFrame(S_sec,index=node_names,columns=node_names)
        h = sns.heatmap(
            S_sec,
            #vmax=17e-7,
            #vmin=0,
            robust=True,
            #center=0,
            linewidths=0.0015,
            linecolor='black',
            cmap=pal,
            cbar= i == 0,
            xticklabels= 3 if i == 11 else "auto",
            yticklabels = 3 if i ==1 else "auto",
            cbar_ax= cbar_ax,
            cbar_kws= None if i else {
                'shrink':1,
                'fraction':0.05,
                'label':cbar_label,
                'drawedges':False
            }, 
            ax = ax,
            rasterized=True
        )
        h.set_yticklabels(h.get_yticklabels(), rotation = 45, fontsize = 6)
        h.set_xticklabels(h.get_xticklabels(), rotation=45, fontsize=6)
        ax.set_title("Secondary {i}".format(i=i+1),fontsize=8)
    
    #Tight layout
    plt.tight_layout(rect=[0, 0, .875, 1])
    #Save figures
    plt.savefig("/home/sam/research/mohca_cl/mohca_cl/ga_tech/secondary/Figures_Secondary/12sec_"+suptitle[:6]+".pdf",dpi=400)
    plt.savefig("/home/sam/research/mohca_cl/mohca_cl/ga_tech/secondary/Figures_Secondary/12sec_"+suptitle[:6]+".png",dpi=400)
    plt.figure()

## Compare the Eigenvalues of the Secondaries

In [ ]:
eigs_tot = np.linalg.eigvals(svp0)
for i,(sec_name,d) in enumerate(secondaries.items()):
    S = d["S"]
    eigmax = lambda A: np.max(np.abs(np.linalg.eigvals(A)))
    eigvals = np.linalg.eigvals(S)
    #print(eigvals)
    print(eigmax(S))
    

In [ ]:
np.max(svq0[33:,33:])

In [ ]:
M = len(vnet)
N = Ynet.shape[0]
r,x = [],[]
for i,y_i in enumerate(Ynet):
    for j, y_ij in enumerate(y_i):
        if j<i and abs(y_ij)>0:
            r.append(np.real(y_ij))
            x.append(np.imag(y_ij))


In [ ]:
def lower_offidag_nonzero(M):
    """
    Construct a vector of the nonzero elements of the lower offdiagonal portion of the matrix M
    """
    elements = []
    for i,m_i in enumerate(M):
        for j, m_ij in enumerate(m_i):
            if j<i and abs(m_ij)>0:
                elements.append(m_ij)
    return np.array(elements)

def get_impedance_matrix(Y):
    X,R = np.zeros_like(Y),np.zeros_like(Y)
    for i,y_i in enumerate(Y):
        for j, y_ij in enumerate(y_i):
            if j<i and abs(y_ij)>0:
                z_ij = 1/y_ij
                X[i,j] = np.real(z_ij)
                R[i,j] = np.imag(z_ij)
    return X + R*j

def get_line_impedances(Y):
    """
    Construct a vector of the nonzero elements of the lower offdiagonal portion of the bus admittance matrix Y
    This results int he line impedances
    """
    Z = get_impedance_matrix(Y)
    X,R = np.real(Z),np.imag(Z)
    xs,rs = lower_offidag_nonzero(X),lower_offidag_nonzero(R)
    return xs,rs


In [ ]:
Znet = get_impedance_matrix(Ynet)

In [ ]:
xs,rs = get_line_impedances(Ynet)

In [ ]:
#Upper and lower off-diagonal elements
def U(A):
    u = []
    for i,ai in enumerate(A):
        for j,aij in enumerate(ai):
            if (i>j):
                u.append(aij)
    return u

def L(A):
    l = []
    for i,ai in enumerate(A):
        for j,aij in enumerate(ai):
            if (i<j):
                l.append(aij)
    return l
def is_radial(Y):  
    n = Y.shape[0]
     
    
    #Get the nonzero upper and lower off diagonal elements
    nz_upper = [1 for y_ij in U(Y) if y_ij != 0]
    nz_lower = [1 for y_ij in L(Y) if y_ij != 0]
    nnnz_u,nnz_l = np.sum(nz_upper),np.sum(nz_lower)
    print(nnnz_u,nnz_l)
    is_not_radial = np.sum(nz_upper)>n-1 or np.sum(nz_lower) >n-1
    is_radial = not(is_not_radial)
    return is_radial


In [ ]:
import cvxpy as cp
sens.compile_dss(cktfile)
sens.dss.run_command("solve")
i = sens.get_node_currents()
v = sens.get_node_voltages()

In [ ]:
s = []
p = []
q = []
vmsq = []
for k,(name,current) in enumerate(i.items()):
    s.append( v[name]*np.conjugate(current))
    p.append(np.real(v[name]*np.conjugate(current)))
    q.append(np.imag(v[name]*np.conjugate(current)))
    vmsq.append(np.abs(v[name])**2)
vmsq,p,q = np.array(vmsq),np.array(p),np.array(q)
s = np.array(s)

## Item 3: Case 3 variants

In [ ]:
case3_unbalanced = r"/home/sam/Research/yadi/test_cases/case3_unbalanced.dss"
case3_unbalanced_delta = r"/home/sam/Research/yadi/test_cases/case3_unbalanced_delta_loads.dss"
case3_unbalanced_1ph = r"/home/sam/Research/yadi/test_cases/case3_unbalanced_1phase-pv.dss"

In [ ]:
case3 = {}
for name,cktfile in zip(["std","delta","1phpv"],[case3_unbalanced,case3_unbalanced_delta,case3_unbalanced_1ph]):
    sens = dss_sens.OpenDSS_Sensitivities(cktfile,verbose=False)
    Ynet,vnet = Y = sens.get_node_ybus(True)
    svp = sens.get_spv()
    svq = sens.get_sqv()
    svp0,svq0 = svp['matrix'],svq['matrix']
    nodes,vbase = svp["nodes"],svp["vph_base"]
    #Store results
    case3[name] = {
        "Y":Ynet,
        "vnet":vnet,
        "vbase":vbase,
        "svp":svp0,
        "svq":svq0,
        "is_radial":is_radial(Ynet)
    }

In [ ]:
case3['std']['Y'][:,3]